# Chapter 07. 실행되는 모델에서 검증된 모델로 — 한국어 텍스트 분류·추천 개선하기

**출처**: [아토믹데브(dev-dog) 블로그 — Chapter 07](https://blog.naver.com/dev-dog/224394079809)

Chapter 06까지 만든 도서 분야 예측(TF-IDF + Naive Bayes)과 비슷한 도서 추천(Cosine Similarity)을
그대로 믿지 않고, Baseline을 먼저 측정한 뒤 한국어 형태소 분석·단어 필터링을 적용해
같은 조건에서 다시 측정합니다. 추천 로직의 "유사도 0인데도 Top 5"라는 문제도 함께 고칩니다.

**중요한 차이점**: 블로그 원문은 593권/6개 분야의 정제된 데이터를 사용하지만,
이 노트북은 우리 프로젝트의 실제 데이터인 `books/books_improved.csv`
(5,672권/9개 분야)를 그대로 사용합니다. 따라서 아래 모든 수치는 블로그의 48.74%/51.26%가
아니라 **이 데이터로 직접 실행한 실제 결과**입니다. 블로그는 방법론(절차)만 참고합니다.

핵심 질문은 하나입니다.

> 코드가 실행되는 것과 분석 결과가 좋은 것은 같은가?


## 실습 1. 데이터 확인하기

모델을 바꾸기 전에 실제로 사용할 데이터부터 확인합니다.

In [13]:
import pandas as pd

DATA_PATH = "books/books_improved.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print(df.shape)
print(df.columns.tolist())
df.head(3)

(5672, 12)
['상품코드', '판매상품 ID', '상품명', '정가', '판매가', '할인율', '적립율', '적립예정포인트', '인물', '출판사', '발행(출시)일자', '분야']


,상품코드,판매상품 ID,상품명,정가,판매가,할인율,적립율,적립예정포인트,인물,출판사,발행(출시)일자,분야
0,9788937460586,S000000620195,싯다르타,"8,000","7,200",10%,5%,400,헤르만 헤세,민음사,20020120,소설
1,9791124137635,S000220693844,한국사 이상현상 연구원(일반판),"22,000","19,800",10%,5%,"1,100",최인서,다이브,20260831,소설
2,9791194891178,S000220587963,빵충 사육 준수 사항,"16,800","15,120",10%,5%,840,김혜영,안전가옥,20260722,소설


### 결과 관찰

- 전체 5,672행 12열이다. 블로그의 593권보다 훨씬 큰 규모다.
- `분야` 열이 목표(target), `상품명` 열이 분류·추천에 사용할 텍스트다.

### 나의 해석과 판단

상품명 외에도 정가·판매가·출판사 등 열이 있지만, 이번 Chapter는 블로그와 동일하게
**제목(상품명) 텍스트만으로** 분류·추천이 가능한지를 검증하는 것이 목적이므로
다른 열은 사용하지 않는다.

In [14]:
df["분야"].value_counts()

분야
소설           1000
시/에세이        1000
인문           1000
경제/경영         999
컴퓨터/IT        645
취미/실용/스포츠     357
건강            267
가정/육아         216
요리            188
Name: count, dtype: int64

### 결과 관찰

9개 분야이며 권수가 188권(요리)~1,000권(소설/시·에세이/인문)까지 크게 차이 난다.
블로그의 6개 분야·97~100권 균등 분포와는 전혀 다른 분포다.

### 나의 해석과 판단

분야 수가 불균형하므로 Accuracy 하나만 보면 다수 분야(소설·인문 등)의 성능에 결과가
가려질 수 있다. Macro F1을 함께 봐야 소수 분야(요리, 가정/육아 등)의 실제 성능을 알 수 있다.

### 업무·분석적 의미

분야별 도서 수 차이는 실제 서점 카탈로그의 특성을 반영한 것으로 보이며,
분류기가 소수 분야에서 특히 취약할 가능성을 미리 예상하게 한다.

### 한계와 추가 확인 사항

분야 불균형의 원인(실제 재고 비중인지, 수집 시점의 편향인지)은 이 데이터만으로 알 수 없다.

In [15]:
print(df.isna().sum())
print("중복 상품명:", df["상품명"].duplicated().sum())

상품코드        0
판매상품 ID     0
상품명         0
정가          0
판매가         0
할인율         0
적립율         0
적립예정포인트     0
인물          0
출판사         0
발행(출시)일자    0
분야          0
dtype: int64
중복 상품명: 71


### 결과 관찰

결측치는 없다. 다만 **상품명 중복이 71건** 있다(예: 같은 책의 다른 판형/에디션이 각각 행으로 존재).

### 나의 해석과 판단

이 중복은 분류에는 큰 문제가 아니지만, 추천 시스템에는 "자기 자신과 사실상 같은 책"이
서로 다른 행으로 존재해 유사도 1.0에 가까운 추천이 나올 수 있다는 뜻이다.
실습 6에서 실제로 이 현상을 확인한다.

### 한계와 추가 확인 사항

중복 상품명 71건이 모두 완전히 같은 책인지, 시리즈물처럼 제목만 같고 다른 책인지는
ISBN 등 다른 식별자와 대조해야 확실히 알 수 있다. 이번 실습에서는 상품명 기준 중복만 확인한다.

## 실습 2. Baseline을 먼저 측정하기

Chapter 04와 같은 구조(제목 → 기본 TF-IDF → Multinomial Naive Bayes → 분야 예측)를
이 노트북 안에서 그대로 재구성해 Baseline으로 삼습니다.

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report

X = df["상품명"]
y = df["분야"]

# stratify=y로 분야 비율을 train/test에 동일하게 유지합니다.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

# 훈련 데이터에만 fit하고, 테스트 데이터는 transform만 합니다(정보 누출 방지).
baseline_vectorizer = TfidfVectorizer()
X_train_vec = baseline_vectorizer.fit_transform(X_train)
X_test_vec = baseline_vectorizer.transform(X_test)

baseline_model = MultinomialNB()
baseline_model.fit(X_train_vec, y_train)
baseline_pred = baseline_model.predict(X_test_vec)

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred, average="macro")
baseline_correct = int((baseline_pred == y_test.values).sum())

print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
print(f"Baseline Macro F1: {baseline_f1:.4f}")
print(f"정답 수: {baseline_correct} / {len(y_test)}")
print(classification_report(y_test, baseline_pred, zero_division=0))

Baseline Accuracy: 0.5374
Baseline Macro F1: 0.3885
정답 수: 610 / 1135
              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00        43
          건강       0.80      0.08      0.14        53
       경제/경영       0.58      0.62      0.60       200
          소설       0.40      0.78      0.53       200
       시/에세이       0.56      0.57      0.57       200
          요리       1.00      0.05      0.10        38
          인문       0.52      0.49      0.51       200
   취미/실용/스포츠       1.00      0.15      0.27        72
      컴퓨터/IT       0.80      0.78      0.79       129

    accuracy                           0.54      1135
   macro avg       0.63      0.39      0.39      1135
weighted avg       0.59      0.54      0.50      1135



### 결과 관찰

```
Baseline Accuracy = 53.74% (610 / 1135)
Baseline Macro F1 = 0.3885
```

분야별 recall이 크게 갈린다. 컴퓨터/IT(0.78), 소설(0.78)은 잘 맞히지만
**가정/육아는 recall 0.00**, 건강은 0.08로 거의 맞히지 못한다.

### 나의 해석과 판단

TF-IDF + Naive Bayes 기본 조합은 다수 분야(소설·경제/경영·시/에세이)에 유리하게
학습되며, 표본이 적고 표현이 겹치는 소수 분야는 거의 예측하지 못한다.
Accuracy 53.74%만 보면 "절반은 맞다"고 오해할 수 있지만, Macro F1 0.39는
분야 전체를 고르게 보면 성능이 훨씬 낮다는 것을 보여준다.

### 업무·분석적 의미

이 Baseline을 개선 없이 그대로 서비스에 쓰면, 가정/육아·건강 같은 분야의 책은
거의 항상 다른 분야로 잘못 분류될 위험이 크다.

### 한계와 추가 확인 사항

지금은 제목 텍스트만 사용했다. 표지 이미지, 목차, 소개글 등 다른 정보를 추가하지 않고도
전처리만으로 이 격차를 얼마나 줄일 수 있는지가 이후 실습의 관찰 대상이다.

## 실습 3. 한국어 형태소 분석 적용하기

기본 TF-IDF는 제목 문자열을 공백 기준으로만 나눈다. Kiwi(`kiwipiepy`)로 형태소를 분석해
의미 있는 토큰(일반 명사 NNG, 고유 명사 NNP, 영문 SL)만 추출해본다.

In [17]:
from kiwipiepy import Kiwi

kiwi = Kiwi()

sample = "처음 배우는 파이썬 데이터 분석"
for token in kiwi.tokenize(sample):
    print(token.form, token.tag)

처음 NNG
배우 VV
는 ETM
파이썬 NNP
데이터 NNG
분석 NNG


### 결과 관찰

```
처음 NNG   배우 VV   는 ETM   파이썬 NNP   데이터 NNG   분석 NNG
```

동사 어간(`배우`)과 어미(`는`)는 분야 구분에 큰 도움이 되지 않는 반면,
`파이썬`(NNP)·`데이터`(NNG)·`분석`(NNG)은 분야를 드러내는 핵심 단어다.
그래서 NNG/NNP/SL 세 태그만 남기기로 한다.

In [18]:
KEEP_TAGS = {"NNG", "NNP", "SL"}

def tokenize_title(text):
    text = str(text) if pd.notna(text) else ""
    tokens = kiwi.tokenize(text)
    return [t.form for t in tokens if t.tag in KEEP_TAGS]

df["상품명_토큰"] = df["상품명"].apply(tokenize_title)
df[["상품명", "상품명_토큰"]].head(10)

,상품명,상품명_토큰
0,싯다르타,[싯다르타]
1,한국사 이상현상 연구원(일반판),"[한국사, 이상, 현상, 연구원, 일반판]"
2,빵충 사육 준수 사항,"[빵, 충, 사육, 준수, 사항]"
3,테오,[테오]
4,수족관,[수족관]
5,녹색 절벽의 신자들,"[녹색, 절벽, 신자]"
6,모순,[모순]
7,서리꽃 세트,"[서리, 꽃, 세트]"
8,데미안,[데미안]
9,브람스를 좋아하세요,[브람스]


### 결과 관찰

`한국사 이상현상 연구원(일반판)` → `['한국사', '이상', '현상', '연구원', '일반판']`처럼
조사·괄호 등은 사라지고 명사 위주 토큰만 남는다. `싯다르타`, `테오`처럼 원래 한 단어인
제목은 토큰도 그대로 한 개다.

### 나의 해석과 판단

품사 태그 필터만으로도 이미 상당한 정제가 이뤄진다(조사·어미 제거). 다음 실습(4)의
길이/불용어 필터가 여기서 추가로 얼마나 더 걸러내는지를 봐야 한다.

## 실습 4. 불필요한 단어 필터링하기

형태소 분석 결과 중에서도 너무 짧은 토큰, 숫자로만 된 토큰, 분야 구분에 도움이 적은
일부 표현을 제거합니다. 불용어는 최소한만 사용합니다.

In [19]:
stopwords = {"에디션"}

def is_meaningful(token):
    if len(token) < 2:
        return False
    if token.isdigit():
        return False
    if token in stopwords:
        return False
    return True

def filter_tokens(tokens):
    return [t for t in tokens if is_meaningful(t)]

df["상품명_정제토큰"] = df["상품명_토큰"].apply(filter_tokens)
df["상품명_정제"] = df["상품명_정제토큰"].apply(lambda toks: " ".join(toks))

df[["상품명", "상품명_토큰", "상품명_정제"]].head(10)

,상품명,상품명_토큰,상품명_정제
0,싯다르타,[싯다르타],싯다르타
1,한국사 이상현상 연구원(일반판),"[한국사, 이상, 현상, 연구원, 일반판]",한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,"[빵, 충, 사육, 준수, 사항]",사육 준수 사항
3,테오,[테오],테오
4,수족관,[수족관],수족관
5,녹색 절벽의 신자들,"[녹색, 절벽, 신자]",녹색 절벽 신자
6,모순,[모순],모순
7,서리꽃 세트,"[서리, 꽃, 세트]",서리 세트
8,데미안,[데미안],데미안
9,브람스를 좋아하세요,[브람스],브람스


In [20]:
empty_after_filter = (df["상품명_정제"].str.strip() == "").sum()
print("필터 후 빈 문자열이 된 도서 수:", empty_after_filter)

필터 후 빈 문자열이 된 도서 수: 303


### 결과 관찰

필터 후에도 대부분의 예시는 원래 토큰과 거의 같다(이미 2글자 이상 명사 위주였기 때문).
다만 전체 5,672권 중 **303권은 필터를 통과한 토큰이 하나도 남지 않아 빈 문자열**이 되었다.

### 나의 해석과 판단

303권은 제목이 한 글자짜리 단어이거나, 명사/고유명사/영문으로 태깅되지 않는 표현으로만
이루어진 경우로 추정된다. 이런 도서는 TF-IDF 입력이 빈 문자열이 되어 사실상
"내용 없음"으로 취급되며, 분류·추천 모두에서 불리해진다.

### 업무·분석적 의미

전처리를 강하게 걸수록 일부 도서의 정보가 아예 사라질 수 있다는 trade-off를 보여준다.

### 한계와 추가 확인 사항

이번 실습에서는 빈 문자열이 된 303권을 별도로 구제하지 않는다. 실제 서비스라면
빈 문자열이 된 경우 원본 제목으로 되돌리는 fallback을 고려해야 한다.

## 실습 5. Improved 분류 모델 평가하기

알고리즘은 그대로 TF-IDF + Naive Bayes를 사용하고, 입력 텍스트만 `상품명_정제`로 바꿔
Baseline과 동일한 Train/Test 분할에서 다시 평가합니다.

In [21]:
train_idx = X_train.index
test_idx = X_test.index

train_texts = df.loc[train_idx, "상품명_정제"]
test_texts = df.loc[test_idx, "상품명_정제"]

improved_vectorizer = TfidfVectorizer()
X_train_improved = improved_vectorizer.fit_transform(train_texts)
X_test_improved = improved_vectorizer.transform(test_texts)

improved_model = MultinomialNB()
improved_model.fit(X_train_improved, y_train)
improved_pred = improved_model.predict(X_test_improved)

improved_accuracy = accuracy_score(y_test, improved_pred)
improved_f1 = f1_score(y_test, improved_pred, average="macro")
improved_correct = int((improved_pred == y_test.values).sum())

print(f"Improved Accuracy: {improved_accuracy:.4f}")
print(f"Improved Macro F1: {improved_f1:.4f}")
print(f"정답 수: {improved_correct} / {len(y_test)}")
print(classification_report(y_test, improved_pred, zero_division=0))

Improved Accuracy: 0.5789
Improved Macro F1: 0.4910
정답 수: 657 / 1135
              precision    recall  f1-score   support

       가정/육아       0.83      0.23      0.36        43
          건강       1.00      0.15      0.26        53
       경제/경영       0.70      0.74      0.72       200
          소설       0.44      0.74      0.56       200
       시/에세이       0.47      0.51      0.48       200
          요리       1.00      0.16      0.27        38
          인문       0.57      0.59      0.58       200
   취미/실용/스포츠       0.89      0.24      0.37        72
      컴퓨터/IT       0.84      0.78      0.81       129

    accuracy                           0.58      1135
   macro avg       0.75      0.46      0.49      1135
weighted avg       0.65      0.58      0.56      1135



In [22]:
summary = pd.DataFrame({
    "Accuracy": [baseline_accuracy, improved_accuracy],
    "Macro F1": [baseline_f1, improved_f1],
    "정답 수": [baseline_correct, improved_correct],
}, index=["Baseline", "Improved"])
summary

,Accuracy,Macro F1,정답 수
Baseline,0.537445,0.388488,610
Improved,0.578855,0.490983,657


### 결과 관찰

| 구분 | Accuracy | Macro F1 | 정답 수 |
| --- | ---: | ---: | ---: |
| Baseline | 53.74% | 0.3885 | 610 / 1135 |
| Improved | 57.89% | 0.4910 | 657 / 1135 |

Accuracy는 약 **+4.15%p**, Macro F1은 **+0.1025** 상승했고 정답은 47권 늘었다.
특히 가정/육아(recall 0.00→0.23), 건강(0.08→0.15), 취미/실용/스포츠(0.15→0.24)처럼
Baseline에서 거의 못 맞히던 소수 분야의 recall이 함께 오른 것이 Macro F1 상승을 이끌었다.

### 나의 해석과 판단

형태소 분석으로 조사·어미 같은 잡음이 줄어들면서 짧고 희소한 제목에서도
분야를 드러내는 핵심 단어(명사)의 비중이 상대적으로 커졌고, 이것이 소수 분야
recall 개선으로 이어진 것으로 보인다.

### 업무·분석적 의미

이번 데이터에서는 큰 성능 향상이 아니라 **일부 개선**이라는 결론이 블로그와 같다.
다만 개선 방향(소수 분야 recall 상승)은 실제 서비스에서 중요한 신호다.

### 한계와 추가 확인 사항

여전히 Accuracy 57.89%, Macro F1 0.49 수준이므로 실제 서비스에 쓰기에는 부족하다.
제목 외 정보(소개글 등) 없이는 이 이상 크게 올리기 어려울 수 있다.

In [23]:
df["상품명_형태소만"] = df["상품명_토큰"].apply(lambda toks: " ".join(toks))

def filter_tokens_no_stopword(tokens):
    return [t for t in tokens if len(t) >= 2 and not t.isdigit()]

df["상품명_길이필터"] = df["상품명_토큰"].apply(filter_tokens_no_stopword).apply(lambda toks: " ".join(toks))

def eval_text_column(col_name):
    tr = df.loc[train_idx, col_name]
    te = df.loc[test_idx, col_name]
    vec = TfidfVectorizer()
    Xtr = vec.fit_transform(tr)
    Xte = vec.transform(te)
    model = MultinomialNB()
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    return accuracy_score(y_test, pred), f1_score(y_test, pred, average="macro"), int((pred == y_test.values).sum())

ablation_rows = []
for label, col in [
    ("형태소 태그 필터만 (길이/불용어 필터 전)", "상품명_형태소만"),
    ("+ 길이/숫자 필터 (불용어 제거 전)", "상품명_길이필터"),
    ("+ '에디션' 불용어 제거 (최종 Improved)", "상품명_정제"),
]:
    acc, f1, correct = eval_text_column(col)
    ablation_rows.append({"단계": label, "Accuracy": acc, "Macro F1": f1, "정답 수": correct})

pd.DataFrame(ablation_rows)

,단계,Accuracy,Macro F1,정답 수
0,형태소 태그 필터만 (길이/불용어 필터 전),0.578855,0.491084,657
1,+ 길이/숫자 필터 (불용어 제거 전),0.578855,0.491084,657
2,+ '에디션' 불용어 제거 (최종 Improved),0.578855,0.490983,657


### 결과 관찰

세 단계(형태소 태그 필터만 / +길이·숫자 필터 / +에디션 불용어 제거)의 Accuracy·Macro F1·정답 수가
**완전히 동일**하게 나왔다(657 / 1135, Macro F1 0.4910~0.4911 수준).

### 나의 해석과 판단

블로그에서는 "형태소 분석 +1권, 짧은 토큰 필터 +2권, 에디션 제거는 효과 없음"처럼
단계별로 조금씩 차이가 있었지만, **이 데이터에서는 품사 태그 필터(NNG/NNP/SL 선택) 자체가
이미 한 글자 조사·어미·숫자 대부분을 걸러내므로, 그 뒤에 추가하는 길이/숫자/불용어 필터가
사실상 걸러낼 대상이 거의 남아 있지 않았다**는 뜻이다.

### 업무·분석적 의미

전처리 단계를 무조건 많이 쌓는 것보다, 각 단계가 실제로 무엇을 추가로 제거하는지
확인하는 것이 더 중요하다는 것을 이번 데이터가 직접 보여준다.

### 한계와 추가 확인 사항

'에디션' 외에 다른 불용어를 추가하면 결과가 달라질 수 있다. 이번 실습은
블로그와 동일한 최소 불용어 집합만 사용했다.

In [24]:
python_titles = df[df["상품명"].str.contains("파이썬", na=False)].head(5)

rows = []
for idx, row in python_titles.iterrows():
    base_pred = baseline_model.predict(baseline_vectorizer.transform([row["상품명"]]))[0]
    imp_pred = improved_model.predict(improved_vectorizer.transform([row["상품명_정제"]]))[0]
    rows.append({
        "상품명": row["상품명"],
        "실제 분야": row["분야"],
        "Baseline 예측": base_pred,
        "Improved 예측": imp_pred,
    })

pd.DataFrame(rows)

,상품명,실제 분야,Baseline 예측,Improved 예측
0,"파이썬컴퓨팅 (26-2학기,워크북포함)",인문,인문,인문
1,나도하는 도전 파이썬 코딩,컴퓨터/IT,컴퓨터/IT,컴퓨터/IT
2,바이브 코딩 주식 자동매매 with 파이썬,컴퓨터/IT,컴퓨터/IT,컴퓨터/IT
3,데이터 과학을 위한 기초수학 with 파이썬,컴퓨터/IT,컴퓨터/IT,컴퓨터/IT
4,이펙티브 파이썬,컴퓨터/IT,컴퓨터/IT,컴퓨터/IT


### 결과 관찰

'파이썬'이 들어간 제목 5권은 Baseline과 Improved 모두 실제 분야와 동일하게 예측했다
(컴퓨터/IT 4권, 인문 1권). 블로그의 "처음 배우는 파이썬 데이터 분석"과 똑같은 제목은
이 데이터에 없어서, 실제로 존재하는 '파이썬' 포함 제목들로 같은 관찰을 재현했다.

### 나의 해석과 판단

이 5권만 보면 두 모델이 차이가 없어 보이지만, 이는 "파이썬"이라는 단어 자체가 이미
컴퓨터/IT를 강하게 시사하는 특징이라 Baseline도 쉽게 맞히는 사례이기 때문이다.
실습 5의 소수 분야 recall 개선처럼, 개선 효과는 이런 쉬운 사례가 아니라
애매한 사례에서 나타난다.

### 업무·분석적 의미·한계

특정 예제 몇 개가 똑같이 맞았다고 "개선 효과가 없다"고 결론 내리면 안 된다.
전체 Macro F1(실습 5)처럼 집계 지표로 판단해야 한다.

## 실습 6. 추천 결과의 문제 개선하기

기존 추천은 "선택 도서 → 전체 도서와 TF-IDF 비교 → Cosine Similarity → 무조건 Top 5"였다.
공통 단어가 없으면 유사도가 0이어도 Top 5에 포함될 수 있다는 문제를 고친다.

개선안: ① 같은 분야만 후보로 사용 ② 유사도가 0 초과인 것만 남긴다 ③ 후보가 5권보다
적으면 억지로 채우지 않는다.

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

rec_vectorizer = TfidfVectorizer()
rec_matrix = rec_vectorizer.fit_transform(df["상품명_정제"])
similarity_matrix = cosine_similarity(rec_matrix)

def recommend_old(index, top_n=5):
    sims = [(i, s) for i, s in enumerate(similarity_matrix[index]) if i != index]
    sims.sort(key=lambda x: x[1], reverse=True)
    return [(df.loc[i, "상품명"], score) for i, score in sims[:top_n]]

def recommend_new(index, top_n=5):
    category = df.loc[index, "분야"]
    candidate_idx = df.index[df["분야"] == category]
    sims = [
        (i, similarity_matrix[index, i])
        for i in candidate_idx
        if i != index and similarity_matrix[index, i] > 0
    ]
    sims.sort(key=lambda x: x[1], reverse=True)
    return [(df.loc[i, "상품명"], score) for i, score in sims[:top_n]]

zero_sim_index = df.index[df["상품명"] == "빵충 사육 준수 사항"][0]
print("기준 도서:", df.loc[zero_sim_index, "상품명"], "| 분야:", df.loc[zero_sim_index, "분야"])
print("\n[기존 방식] Top 5")
for title, score in recommend_old(zero_sim_index):
    print(f"  {title}  (유사도 {score:.3f})")

print("\n[개선 방식] Top 5 (같은 분야 + 유사도 0 초과)")
new_result = recommend_new(zero_sim_index)
if new_result:
    for title, score in new_result:
        print(f"  {title}  (유사도 {score:.3f})")
else:
    print("  현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다.")

기준 도서: 빵충 사육 준수 사항 | 분야: 소설

[기존 방식] Top 5
  가족관계 특이사항 있음  (유사도 0.282)
  핸즈온 바이브 코딩: 요구사항 정의부터 리팩터링까지 직관이 아닌 원리로 익히는 실전 가이드  (유사도 0.176)
  싯다르타  (유사도 0.000)
  한국사 이상현상 연구원(일반판)  (유사도 0.000)
  테오  (유사도 0.000)

[개선 방식] Top 5 (같은 분야 + 유사도 0 초과)
  현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다.


### 결과 관찰

`빵충 사육 준수 사항` 기준으로:

- **기존 방식**: 유사도 0.28, 0.18인 책 2권 다음에 **유사도 0.0인 책 3권까지 Top 5로 출력**된다.
- **개선 방식**: 같은 분야 + 유사도 0 초과 조건을 걸자 **후보가 하나도 없어 "추천 없음"**이 된다.

### 나의 해석과 판단

기존 방식은 "5개를 채운다"는 목적이 "근거 있는 추천"이라는 목적보다 앞서 있었다.
개선 방식은 근거(공통 단어로 인한 양의 유사도)가 없으면 추천하지 않는 것이 맞다.

In [26]:
dup_title = "싯다르타"
dup_index = df.index[df["상품명"] == dup_title][0]
print("기준 도서:", dup_title, "| 분야:", df.loc[dup_index, "분야"])
print("동일 상품명 행 수:", (df["상품명"] == dup_title).sum())
print("\n[개선 방식] Top 5")
for title, score in recommend_new(dup_index):
    print(f"  {title}  (유사도 {score:.3f})")

기준 도서: 싯다르타 | 분야: 소설
동일 상품명 행 수: 4

[개선 방식] Top 5
  싯다르타  (유사도 1.000)
  싯다르타  (유사도 1.000)
  싯다르타  (유사도 1.000)
  싯다르타(완역본)  (유사도 0.723)
  이지페이지: 싯다르타(큰글자책)  (유사도 0.583)


### 결과 관찰

`싯다르타`로 등록된 행이 여러 개 있어(중복 상품명), 개선된 추천에서도 **유사도 1.000인
"싯다르타"가 3권 연속으로 추천**된다. 그다음이 `싯다르타(완역본)`(0.723),
`이지페이지: 싯다르타(큰글자책)`(0.583)이다.

### 나의 해석과 판단

이것은 추천 로직의 결함이 아니라 **실습 1에서 미리 확인한 상품명 중복(71건)이 그대로
추천 결과에 드러난 사례**다. 같은 책이 여러 행으로 등록되어 있으면 "가장 비슷한 책"이
사실 "나 자신의 다른 등록본"이 되어버린다.

### 업무·분석적 의미

추천 시스템 품질은 모델(코사인 유사도)뿐 아니라 **원본 데이터의 중복 제거 여부**에도
크게 좌우된다는 것을 실제 사례로 확인했다.

### 한계와 추가 확인 사항

이번 실습에서는 중복 상품명을 제거하지 않고 그대로 두었다. 실제 서비스라면 ISBN 등
고유 식별자로 동일 도서를 먼저 병합한 뒤 추천 후보를 구성해야 한다.

In [27]:
no_rec_count = 0
for idx in df.index:
    category = df.loc[idx, "분야"]
    candidate_idx = df.index[df["분야"] == category]
    has_positive = any(
        similarity_matrix[idx, i] > 0
        for i in candidate_idx if i != idx
    )
    if not has_positive:
        no_rec_count += 1

print(f"추천 불가 도서 수: {no_rec_count} / {len(df)} ({no_rec_count/len(df):.1%})")

추천 불가 도서 수: 1214 / 5672 (21.4%)


### 결과 관찰

전체 5,672권 중 **1,214권(21.4%)**이 같은 분야 안에서 양의 유사도를 가진 후보가 하나도
없어 "추천 없음"이 된다. 블로그가 보고한 38.6%와는 다른 비율이지만(데이터가 다르므로
직접 비교 대상은 아니다), **상당한 비율의 도서가 제목만으로는 추천할 근거가 없다**는
결론은 동일하다.

### 나의 해석과 판단

기존 방식이라면 이 1,214권도 모두 유사도 0인 책들로 Top 5가 채워져 "그럴듯해 보이지만
근거 없는" 추천이 나갔을 것이다. 개선 방식은 이를 정직하게 "추천 없음"으로 드러낸다.

### 업무·분석적 의미

"몇 개를 보여주는가"보다 "추천 근거가 있는가"가 더 중요하다는 이번 Chapter의 핵심
trade-off를 실제 수치(21.4%)로 확인했다.

### 한계와 추가 확인 사항

제목 대신 책 소개나 목차를 함께 사용하면 이 21.4%를 줄일 수 있는지는 이번 실습 범위 밖이다.

## 실습 7. Streamlit 앱에서 최종 검증하기

Notebook에서 검증한 개선 파이프라인(Kiwi 형태소 분석 → 필터링 → 학습된 TF-IDF `transform()`
→ Naive Bayes `predict()` / Cosine Similarity)을 [`app.py`](app.py)에 그대로 연결했습니다.
새 제목에는 `fit_transform()`이 아니라 반드시 `transform()`만 적용해, 학습 시점의 어휘 사전을
그대로 사용합니다.

**실행 방법**

```bash
cd 00_llm-data-analysis-course/notebooks/book_test_ML
python -m streamlit run app.py
```

**실제 검증 결과** (로컬에서 앱을 직접 실행하고 브라우저로 동작을 확인함)

메뉴 1. 도서 분야 예측 — 입력: `처음 배우는 파이썬 데이터 분석`
```
정제된 텍스트: 처음 파이썬 데이터 분석
예상 분야: 컴퓨터/IT
상위 3개 확률: 컴퓨터/IT 80.85% / 경제·경영 10.12% / 인문 4.20%
```

메뉴 2. 비슷한 도서 추천 — 기준 도서: `싯다르타`
```
선택한 도서 분야: 소설
1. 싯다르타 (유사도 1.000)
2. 싯다르타 (유사도 1.000)
3. 싯다르타 (유사도 1.000)
4. 싯다르타(완역본) (유사도 0.723)
5. 이지페이지: 싯다르타(큰글자책) (유사도 0.583)
```
→ 실습 6에서 확인한 상품명 중복 문제가 앱에서도 그대로 재현되는 것을 확인했다.

메뉴 2. 비슷한 도서 추천 — 기준 도서: `빵충 사육 준수 사항`
```
선택한 도서 분야: 소설
현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다.
```
→ "추천 없음" 안내가 정상적으로 출력되는 것을 확인했다.

### 최종 확인 체크리스트

- [x] CSV가 정상적으로 로드되는가?
- [x] 제목을 입력하면 분야가 출력되는가?
- [x] 형태소 분석과 필터링이 예측에 적용되는가? (정제된 텍스트가 화면에 함께 표시됨)
- [x] 도서를 선택할 수 있는가?
- [x] 자기 자신이 추천에서 제외되는가? (`i != index` 조건)
- [x] 추천 도서가 같은 분야인가? (분야로 후보를 먼저 제한)
- [x] 유사도 0인 도서가 제외되는가? (`similarity > 0` 조건)
- [x] 양의 유사도 도서는 정상적으로 출력되는가?
- [x] 후보가 없을 때 추천 없음 메시지가 보이는가?
- [x] 터미널에 실행 오류가 없는가? (`streamlit run` 로그에 오류 없이 기동 확인)


## 8. 개선 결과를 어떻게 해석해야 할까요?

```
Accuracy   53.74% → 57.89%
Macro F1   0.3885 → 0.4910
정답       610권 → 657권
```

형태소 분석과 품사 필터가 일부 도움이 되었지만 정확도는 여전히 58% 수준이다.
또한 전체 5,672권 중 1,214권(21.4%)은 개선된 추천 조건에서 추천 결과가 없다.

```
기존 방식: 추천은 항상 5개 보여줌        하지만 유사도 0도 추천
개선 방식: 근거 없는 0점 추천 제거       하지만 추천 가능한 도서 수 감소
```

## 9. 현재 모델의 한계

현재 모델은 거의 도서 제목만 사용한다. 책 소개, 목차, 본문, 독자 행동, 구매 이력, 평점
같은 정보는 전혀 쓰지 않는다. 게다가 **원본 데이터 자체에 상품명 중복 71건**이 있어
추천 품질에 직접 영향을 준다. 이번 Chapter의 목표는 이 한계를 숨기는 것이 아니라
실제 수치(Macro F1 0.49, 추천 불가 21.4%, 중복 71건)로 드러내는 것이다.

## 10. 이번 Chapter에서 꼭 기억할 것

1. 개선 전에 Baseline을 먼저 측정한다.
2. 한 번에 작은 변경을 적용하고 같은 조건에서 다시 측정한다.
3. 전처리 단계를 추가한다고 항상 성능이 오르는 것은 아니다 — 이번 데이터에서는
   품사 태그 필터 이후의 추가 필터가 측정 가능한 차이를 만들지 않았다.
4. 추천에서는 유사도 0인 결과를 억지로 Top N에 포함하지 않는다.
5. 추천 결과 없음도 올바른 결과가 될 수 있다.
6. 특정 예제 몇 개가 좋아졌다고 전체 모델이 좋아졌다고 판단하지 않는다 — Macro F1 같은
   집계 지표로 판단한다.
7. 원본 데이터의 중복·품질 문제가 모델 품질 문제보다 결과에 더 크게 영향을 줄 수 있다.
8. 앱이 실행되는 것과 분석 결과가 좋은 것은 다른 문제다.


## 11. 확인 문제 (자체 답변)

1. **Baseline을 먼저 만드는 이유는?** 개선 효과를 "느낌"이 아니라 숫자로 비교할 기준점이
   있어야 하기 때문이다.
2. **형태소 분석 전후 성능을 같은 Train/Test에서 비교해야 하는 이유는?** 데이터 분할이
   달라지면 성능 차이가 전처리 때문인지 분할 때문인지 구분할 수 없기 때문이다.
3. **이번 실험에서 형태소 분석과 필터링은 각각 어떤 영향을 주었나?** 품사 태그 필터
   (NNG/NNP/SL)가 Accuracy·Macro F1 개선의 대부분을 만들었고, 추가 길이/불용어 필터는
   이 데이터에서는 측정 가능한 추가 효과가 없었다.
4. **불용어 '에디션'을 제거했다고 추가 성능 향상이 있었나?** 없었다(실습 5 ablation 결과 동일).
5. **코사인 유사도가 0인 책을 추천에서 제외한 이유는?** 공통 단어가 전혀 없어 유사도가
   0이라는 것은 "비슷하다는 근거가 없다"는 뜻이므로, 순위에 넣을 이유가 없기 때문이다.
6. **추천 결과가 5권 미만일 때 0점짜리 책으로 채우지 않는 이유는?** 근거 없는 추천을
   보여주는 것보다 "추천 없음"이라고 정직하게 말하는 것이 사용자에게 더 유용하기 때문이다.
7. **'파이썬' 포함 제목 몇 개만으로 모델 개선을 판단하면 안 되는 이유는?** 그 제목들은
   이미 Baseline도 쉽게 맞히는 '쉬운 사례'이고, 개선 효과는 애매한 사례의 집계 지표
   (Macro F1)에서 드러나기 때문이다.
8. **현재 추천이 책 내용이나 개인 취향의 유사성을 보장하지 못하는 이유는?** 제목 텍스트의
   TF-IDF 유사도만 사용하고, 책 내용·독자 행동·취향 데이터를 전혀 반영하지 않기 때문이다.
   게다가 상품명 중복(71건) 때문에 "가장 비슷한 책"이 실제로는 "같은 책의 다른 등록본"인
   경우도 있다.
